# Assignment 2.1: Data Lake Exercise
Name: Ali Azizi

Dataset source:
https://github.com/mechristenson/aai-540-homework/blob/main/homework-2-1/data/dataset.csv

Goal:
Ingest the homework Spotify dataset into an S3 data lake, create an Athena table, and answer the required questions using SQL and Pandas.

In [1]:
# Setup imports
import boto3
import sagemaker
import pandas as pd 
from pyathena import connect

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [4]:
sess = sagemaker.Session()
bucket = sess.default_bucket()
region = boto3.Session().region_name

print("Bucket: ", bucket)
print("region: ", region)

Bucket:  sagemaker-us-east-1-646180330281
region:  us-east-1


In [5]:
dataset_url = "https://raw.githubusercontent.com/mechristenson/aai-540-homework/main/homework-2-1/data/dataset.csv"

df_local = pd.read_csv(dataset_url)
df_local.head()

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [6]:
df_local.shape

(114000, 21)

In [7]:
df_local.columns

Index(['Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name',
       'popularity', 'duration_ms', 'explicit', 'danceability', 'energy',
       'key', 'loudness', 'mode', 'speechiness', 'acousticness',
       'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature',
       'track_genre'],
      dtype='object')

In [8]:
# Saving the dataset locally
local_file = "dataset.csv"
df_local.to_csv(local_file, index=False)
print("saved:", local_file)

saved: dataset.csv


# Upload CSV to the S3 data lake

In [9]:
s3_prefix = "homework-2-1/spotify-csv"
s3_csv_path = f"s3://{bucket}/{s3_prefix}/dataset.csv"

!aws s3 cp dataset.csv {s3_csv_path}


upload: ./dataset.csv to s3://sagemaker-us-east-1-646180330281/homework-2-1/spotify-csv/dataset.csv


In [10]:
# Checking
!aws s3 ls s3://{bucket}/{s3_prefix}/

2026-05-20 05:48:05   20118254 dataset.csv


In [11]:
bucket = sess.default_bucket()
print(bucket)

sagemaker-us-east-1-646180330281


# Creating Athena database
Like the lab creating an Athena database first, then registers tables in Glue/Athena, we do the same!

In [12]:
database_name = "aai540_assignment2"
s3_staging_dir = f"s3://{bucket}/athena/staging"

conn = connect(region_name=region, s3_staging_dir=s3_staging_dir)

statement = f"CREATE DATABASE IF NOT EXISTS {database_name}"
pd.read_sql(statement, conn)

/tmp/ipykernel_12002/2830262516.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(statement, conn)


""


In [13]:
pd.read_sql("SHOW DATABASES", conn)

/tmp/ipykernel_12002/2453565319.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql("SHOW DATABASES", conn)


,database_name
0,aai540_assignment2
1,default
2,dsoaws


# Creating Athena CSV table
Because our dataset is a CSV file, we use OpenCSVSerde.

In [14]:
table_name_csv = "spotify_tracks_csv"
s3_csv_folder = f"s3://{bucket}/{s3_prefix}/"

statement = """CREATE EXTERNAL TABLE IF NOT EXISTS {}.{}(
    song_index int,
    track_id string,
    artists string,
    album_name string,
    track_name string,
    popularity int,
    duration_ms int,
    explicit boolean,
    danceability double,
    energy double,
    key int,
    loudness double,
    mode int,
    speechiness double,
    acousticness double,
    instrumentalness double,
    liveness double,
    valence double,
    tempo double,
    time_signature int,
    track_genre string
)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
WITH SERDEPROPERTIES (
    "separatorChar" = ",",
    "quoteChar" = "\\"",
    "escapeChar" = "\\\\"
)
LOCATION '{}'
TBLPROPERTIES ('skip.header.line.count'='1')""".format(
    database_name, table_name_csv, s3_csv_folder
)

print(statement)
pd.read_sql(statement, conn)

CREATE EXTERNAL TABLE IF NOT EXISTS aai540_assignment2.spotify_tracks_csv(
    song_index int,
    track_id string,
    artists string,
    album_name string,
    track_name string,
    popularity int,
    duration_ms int,
    explicit boolean,
    danceability double,
    energy double,
    key int,
    loudness double,
    mode int,
    speechiness double,
    acousticness double,
    instrumentalness double,
    liveness double,
    valence double,
    tempo double,
    time_signature int,
    track_genre string
)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
WITH SERDEPROPERTIES (
    "separatorChar" = ",",
    "quoteChar" = "\"",
    "escapeChar" = "\\"
)
LOCATION 's3://sagemaker-us-east-1-646180330281/homework-2-1/spotify-csv/'
TBLPROPERTIES ('skip.header.line.count'='1')


/tmp/ipykernel_12002/4093269600.py:39: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(statement, conn)


""


In [15]:
pd.read_sql(f"SHOW TABLES IN {database_name}", conn)

/tmp/ipykernel_12002/1164958132.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(f"SHOW TABLES IN {database_name}", conn)


,tab_name
0,spotify_tracks_csv


In [16]:
statement = f"SELECT * FROM {database_name}.{table_name_csv} LIMIT 10"
df_sql_preview = pd.read_sql(statement, conn)
df_sql_preview

/tmp/ipykernel_12002/469621367.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_sql_preview = pd.read_sql(statement, conn)


,song_index,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.7150,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.2670,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.1200,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.1430,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.1670,119.949,4,acoustic
5,5,01MVOl9KtVTNfFiBU9I7dc,Tyrone Wells,Days I Will Remember,Days I Will Remember,58,214240,False,0.688,0.4810,...,-8.807,1,0.1050,0.2890,0.000000,0.1890,0.6660,98.017,4,acoustic
6,6,6Vc5wAMmXdKIAM7WUoEb7N,A Great Big World;Christina Aguilera,Is There Anybody Out There?,Say Something,74,229400,False,0.407,0.1470,...,-8.822,1,0.0355,0.8570,0.000003,0.0913,0.0765,141.284,3,acoustic
7,7,1EzrEOXmMH3G43AXT1y7pA,Jason Mraz,We Sing. We Dance. We Steal Things.,I'm Yours,80,242946,False,0.703,0.4440,...,-9.331,1,0.0417,0.5590,0.000000,0.0973,0.7120,150.960,4,acoustic
8,8,0IktbUcnAGrvD03AWnz3Q8,Jason Mraz;Colbie Caillat,We Sing. We Dance. We Steal Things.,Lucky,74,189613,False,0.625,0.4140,...,-8.700,1,0.0369,0.2940,0.000000,0.1510,0.6690,130.088,4,acoustic
9,9,7k9GuJYLp2AzqokyEdwEw2,Ross Copperman,Hunger,Hunger,56,205594,False,0.442,0.6320,...,-6.770,1,0.0295,0.4260,0.004190,0.0735,0.1960,78.899,4,acoustic


# Query 1: artist, track_name, popularity where popularity >= 99

## SQL Answer:

In [17]:
statement = f"""
SELECT artists, track_name, popularity
FROM {database_name}.{table_name_csv}
WHERE popularity >= 99
ORDER BY popularity DESC
"""

df_q1_sql = pd.read_sql(statement, conn)
df_q1_sql

/tmp/ipykernel_12002/3562346582.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_q1_sql = pd.read_sql(statement, conn)


,artists,track_name,popularity
0,Sam Smith;Kim Petras,Unholy (feat. Kim Petras),100
1,Sam Smith;Kim Petras,Unholy (feat. Kim Petras),100
2,Bizarrap;Quevedo,"Quevedo: Bzrp Music Sessions, Vol. 52",99


## Panda Answer

In [18]:
df_q1_pandas = df_local[df_local["popularity"] >= 99][["artists", "track_name", "popularity"]]
df_q1_pandas.sort_values(by="popularity", ascending=False)

,artists,track_name,popularity
20001,Sam Smith;Kim Petras,Unholy (feat. Kim Petras),100
81051,Sam Smith;Kim Petras,Unholy (feat. Kim Petras),100
51664,Bizarrap;Quevedo,"Quevedo: Bzrp Music Sessions, Vol. 52",99


# Query 2: artists with average popularity of 92
I use rounded average popularity = 92, because exact averages may be decimals.


## SQl Answer:

In [20]:
statement = f"""
SELECT artists, AVG(popularity) AS avg_popularity
FROM {database_name}.{table_name_csv}
GROUP BY artists
HAVING ROUND(AVG(popularity)) = 92
ORDER BY avg_popularity DESC
"""

df_q2_sql = pd.read_sql(statement, conn)
df_q2_sql

/tmp/ipykernel_12002/3050123666.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_q2_sql = pd.read_sql(statement, conn)


,artists,avg_popularity
0,Harry Styles,92.0
1,Rema;Selena Gomez,92.0


In [ ]:
## Pandas

In [21]:
df_q2_pandas = (
    df_local
    .groupby("artists", as_index=False)["popularity"]
    .mean()
    .rename(columns={"popularity": "avg_popularity"})
)

df_q2_pandas[df_q2_pandas["avg_popularity"].round() == 92].sort_values(
    by="avg_popularity", ascending=False
)

,artists,avg_popularity
11491,Harry Styles,92.0
22845,Rema;Selena Gomez,92.0


# Query 3: Top 10 genres with highest average energy

## SQL

In [22]:
statement = f"""
SELECT track_genre, AVG(energy) AS avg_energy
FROM {database_name}.{table_name_csv}
GROUP BY track_genre
ORDER BY avg_energy DESC
LIMIT 10
"""

df_q3_sql = pd.read_sql(statement, conn)
df_q3_sql

/tmp/ipykernel_12002/4283438757.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_q3_sql = pd.read_sql(statement, conn)


,track_genre,avg_energy
0,death-metal,0.931470
1,grindcore,0.924201
2,metalcore,0.914485
3,happy,0.910971
4,hardstyle,0.901246
5,drum-and-bass,0.876635
6,black-metal,0.874897
7,heavy-metal,0.874003
8,party,0.871237
9,j-idol,0.868677


## Pandas

In [23]:
df_q3_pandas = (
    df_local
    .groupby("track_genre", as_index=False)["energy"]
    .mean()
    .rename(columns={"energy": "avg_energy"})
    .sort_values(by="avg_energy", ascending=False)
    .head(10)
)

df_q3_pandas

,track_genre,avg_energy
22,death-metal,0.931470
42,grindcore,0.924201
72,metalcore,0.914485
46,happy,0.910971
49,hardstyle,0.901246
27,drum-and-bass,0.876635
6,black-metal,0.874897
50,heavy-metal,0.874003
78,party,0.871237
61,j-idol,0.868677


## Query 4: How many tracks is Bad Bunny on?

## SQL

In [24]:
statement = f"""
SELECT COUNT(*) AS bad_bunny_track_count
FROM {database_name}.{table_name_csv}
WHERE LOWER(artists) LIKE '%bad bunny%'
"""

df_q4_sql = pd.read_sql(statement, conn)
df_q4_sql

/tmp/ipykernel_12002/314747759.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_q4_sql = pd.read_sql(statement, conn)


,bad_bunny_track_count
0,416


## Pandas

In [25]:
df_q4_pandas = df_local[df_local["artists"].str.lower().str.contains("bad bunny", na=False)]
bad_bunny_track_count = len(df_q4_pandas)

bad_bunny_track_count

416

# Query 5: Top 10 genres in terms of popularity, sorted by their most popular track

In [ ]:
## SQL

In [26]:
statement = f"""
SELECT track_genre, MAX(popularity) AS most_popular_track
FROM {database_name}.{table_name_csv}
GROUP BY track_genre
ORDER BY most_popular_track DESC
LIMIT 10
"""

df_q5_sql = pd.read_sql(statement, conn)
df_q5_sql

/tmp/ipykernel_12002/2784917716.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_q5_sql = pd.read_sql(statement, conn)


,track_genre,most_popular_track
0,dance,100
1,pop,100
2,hip-hop,99
3,reggaeton,98
4,latino,98
5,reggae,98
6,latin,98
7,edm,98
8,piano,96
9,rock,96


## Pandas

In [27]:
df_q5_pandas = (
    df_local
    .groupby("track_genre", as_index=False)["popularity"]
    .max()
    .rename(columns={"popularity": "most_popular_track"})
    .sort_values(by="most_popular_track", ascending=False)
    .head(10)
)

df_q5_pandas

,track_genre,most_popular_track
80,pop,100
20,dance,100
51,hip-hop,99
68,latino,98
89,reggaeton,98
30,edm,98
67,latin,98
88,reggae,98
79,piano,96
90,rock,96
